# 04 — Vector derivatives and least squares ($A\mathbf{x}=\mathbf{b}$)

**Needed for the assignment:** $\bigstar$ YES (background). **Study:** 1.5 h, **Practice:** 1 h.

In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

## Idea

With more equations than unknowns ($m>n$) and noisy data, $A\mathbf{x}=\mathbf{b}$ has no exact solution. Least squares picks the $\mathbf{x}$ that makes the squared error $\|A\mathbf{x}-\mathbf{b}\|^2$ smallest. To find it, take the derivative of a scalar with respect to a vector and set it to zero; this needs three rules.

## Key formulas

**Derivative rules** (course handout "matrix.pdf"): $$\frac{\partial\,\mathbf{a}^T\mathbf{x}}{\partial\mathbf{x}}=\mathbf{a},\qquad
\frac{\partial\,\mathbf{x}^T\mathbf{x}}{\partial\mathbf{x}}=2\mathbf{x},\qquad
\frac{\partial\,\mathbf{x}^TB\mathbf{x}}{\partial\mathbf{x}}=(B+B^T)\mathbf{x}\ \ (=2B\mathbf{x}\text{ if }B\text{ symmetric})$$ **Least squares:** $$J=\|A\mathbf{x}-\mathbf{b}\|^2=\mathbf{x}^TA^TA\mathbf{x}-2\mathbf{x}^TA^T\mathbf{b}+\mathbf{b}^T\mathbf{b}
\ \Rightarrow\ \frac{\partial J}{\partial\mathbf{x}}=2A^TA\mathbf{x}-2A^T\mathbf{b}=\mathbf{0}$$ $$\boxed{A^TA\,\mathbf{x}=A^T\mathbf{b}}\quad(\text{normal equation}),\qquad \mathbf{x}=(A^TA)^{-1}A^T\mathbf{b}$$ Line $y=mx+b$: one row $[x_i,\ 1]$ per point. Plane $z=ax+by+c$: one row $[x_i,\ y_i,\ 1]$ per point.

# Worked example (on paper)

Fit $y=mx+b$ to $(0,1),(1,3),(2,4),(3,4)$.

**1. Build the system.** $A=\begin{bmatrix}0&1\\1&1\\2&1\\3&1\end{bmatrix},\ \mathbf{b}=\begin{bmatrix}1\\3\\4\\4\end{bmatrix}$.

**2. The sums.** $\sum x_i=6,\ \sum x_i^2=14,\ \sum y_i=12,\ \sum x_iy_i=0+3+8+12=23,\ N=4$: $$A^TA=\begin{bmatrix}\sum x^2&\sum x\\\sum x&N\end{bmatrix}=\begin{bmatrix}14&6\\6&4\end{bmatrix},\qquad
A^T\mathbf{b}=\begin{bmatrix}\sum xy\\\sum y\end{bmatrix}=\begin{bmatrix}23\\12\end{bmatrix}$$

**3. Solve the $2\times2$ system.** $\det=14\cdot4-6\cdot6=20$: $$\begin{bmatrix}m\\b\end{bmatrix}=\frac1{20}\begin{bmatrix}4&-6\\-6&14\end{bmatrix}\begin{bmatrix}23\\12\end{bmatrix}
=\frac1{20}\begin{bmatrix}92-72\\-138+168\end{bmatrix}=\begin{bmatrix}1\\1.5\end{bmatrix}$$

**4. Residuals.** $y_i-(x_i+1.5)$: $-0.5,\ 0.5,\ 0.5,\ -0.5$. Sum of squares $=1$. No other line does better.

# The same in Python / NumPy

In [2]:
import numpy as np

# data points
x = np.array([0.0, 1.0, 2.0, 3.0])
y = np.array([1.0, 3.0, 4.0, 4.0])

# A [m, b]^T = y   (one row [x_i, 1] per point)
A = np.column_stack([x, np.ones_like(x)])
print("A =\n", A)
print("A^T A =\n", A.T @ A)
print("A^T y =", A.T @ y)

# normal equation: (A^T A) p = A^T y
p = np.linalg.inv(A.T @ A) @ (A.T @ y)
m, b = p
print("m =", round(m, 4), " b =", round(b, 4))

residual = y - (m * x + b)
print("residuals =", residual, " sum of squares =", np.sum(residual**2))

# the same with NumPy's least-squares solver (more stable than inv)
p2, *_ = np.linalg.lstsq(A, y, rcond=None)
print("lstsq     =", np.round(p2, 4))

# gradient check of J(x) = x^T B x  ->  (B + B^T) x
B = np.array([[1.0, 2.0], [0.0, 3.0]])
x0 = np.array([1.0, -1.0])
J = lambda v: v @ B @ v
eps = 1e-6
num_grad = np.array([(J(x0 + eps * e) - J(x0 - eps * e)) / (2 * eps) for e in np.eye(2)])
print("numeric grad  =", np.round(num_grad, 6))
print("(B + B^T) x0  =", (B + B.T) @ x0)

A =
 [[0. 1.]
 [1. 1.]
 [2. 1.]
 [3. 1.]]
A^T A =
 [[14.  6.]
 [ 6.  4.]]
A^T y = [23. 12.]
m = 1.0  b = 1.5
residuals = [-0.5  0.5  0.5 -0.5]  sum of squares = 1.0000000000000004
lstsq     = [1.  1.5]
numeric grad  = [ 0. -4.]
(B + B^T) x0  = [ 0. -4.]


## Limit of the explicit form

$y=mx+b$ cannot represent a vertical line, and $z=ax+by+c$ cannot represent a vertical wall. It also measures error only along $y$ (or $z$), not perpendicular to the line. For a point cloud with walls in all directions you need the **homogeneous** form $ax+by+cz+d=0$, solved as $A\mathbf{h}=\mathbf{0}$. That is topic 05.

## Where this is used in the assignment

The assignment uses the homogeneous version, so this topic is background. The derivative rules are what you use in topic 05 to derive "smallest eigenvector".

## Practice

Write your solution in the empty cell under each exercise. The **expected result** tells you what you should get, not how to get it. Grey *provided data* cells are already filled in; run them first.

### Exercise 1

Fit $y=mx+b$ to $(0,0),(1,1),(2,1),(3,2)$ by hand with the sums, then with NumPy.

> **Expected result:** $m=0.6$, $b=0.1$.

In [3]:
# Exercise 1: your solution


### Exercise 2

Use the rules to find $\partial J/\partial\mathbf x$ for $J=(\mathbf x-\mathbf a)^T(\mathbf x-\mathbf a)$. Where is the minimum?

> **Expected result:** $\partial J/\partial\mathbf x=2(\mathbf x-\mathbf a)$, so the minimum is at $\mathbf x=\mathbf a$.

In [4]:
# Exercise 2: your solution


### Exercise 3

Fit the plane $z=ax+by+c$ to $(0,0,1),(1,0,3),(0,1,0),(1,1,2),(2,1,4)$ with NumPy.

> **Expected result:** $a=2,\ b=-1,\ c=1$ (these points fit it exactly).

In [5]:
# Exercise 3: your solution


### Exercise 4

Try to fit $y=mx+b$ to the vertical points $(2,0),(2,1),(2,3)$. Compute $A^TA$ and its determinant. What happens with `np.linalg.inv`?

> **Expected result:** $A^TA=\begin{bmatrix}12&6\\6&3\end{bmatrix}$, $\det=0$, and `inv` raises `LinAlgError: Singular matrix`. The explicit form cannot represent a vertical line.

In [6]:
# Exercise 4: your solution
